In [28]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import polars as pl
import numpy as np

df = pl.read_parquet("../data/processed/train.parquet")

print(df.shape)
print(df["session"].n_unique())

(5227653, 4)
100000


In [3]:
df = df.sort(["session", "ts"])

In [4]:
df_val = (
    df
    .with_columns(
        pl.int_range(0, pl.len())
        .over("session")
        .alias("event_pos"),

        pl.len()
        .over("session")
        .alias("session_len")
    )
    .with_columns(
        (
            pl.col("event_pos")
            < (pl.col("session_len") * 0.7).floor()
        ).alias("is_observed")
    )
)

df_val.select(
    "session",
    "aid",
    "type",
    "event_pos",
    "session_len",
    "is_observed"
).head(20)

session,aid,type,event_pos,session_len,is_observed
i64,i64,str,i64,u32,bool
0,1517085,"""clicks""",0,276,true
0,1563459,"""clicks""",1,276,true
0,1309446,"""clicks""",2,276,true
0,16246,"""clicks""",3,276,true
0,1781822,"""clicks""",4,276,true
…,…,…,…,…,…
0,803544,"""clicks""",15,276,true
0,1110941,"""clicks""",16,276,true
0,1190046,"""clicks""",17,276,true


In [5]:
observed = (
    df_val
    .filter(pl.col("is_observed"))
)

observed.head()

session,aid,ts,type,event_pos,session_len,is_observed
i64,i64,i64,str,i64,u32,bool
0,1517085,1659304800025,"""clicks""",0,276,true
0,1563459,1659304904511,"""clicks""",1,276,true
0,1309446,1659367439426,"""clicks""",2,276,true
0,16246,1659367719997,"""clicks""",3,276,true
0,1781822,1659367871344,"""clicks""",4,276,true


In [6]:
hidden = (
    df_val
    .filter(~pl.col("is_observed"))
)

ground_truth = (
    hidden
    .group_by("session")
    .agg(
        pl.col("aid")
        .filter(pl.col("type") == "clicks")
        .first()
        .alias("click_target"),

        pl.col("aid")
        .filter(pl.col("type") == "carts")
        .unique()
        .alias("cart_targets"),

        pl.col("aid")
        .filter(pl.col("type") == "orders")
        .unique()
        .alias("order_targets")
    )
)

ground_truth.head()

session,click_target,cart_targets,order_targets
i64,i64,list[i64],list[i64]
0,190818,"[275288, 974651, … 315914]","[1199474, 543308]"
1,105393,[105393],[]
2,1605583,[],[]
3,822461,"[812246, 1638009, … 925352]","[1018433, 54857]"
4,758750,"[917213, 758750]",[]


In [7]:
example_session = ground_truth["session"][0]

print("Observed:")
display(
    observed
    .filter(pl.col("session") == example_session)
)

print("Hidden:")
display(
    hidden
    .filter(pl.col("session") == example_session)
)

print("Ground truth:")
display(
    ground_truth
    .filter(pl.col("session") == example_session)
)

Observed:


session,aid,ts,type,event_pos,session_len,is_observed
i64,i64,i64,str,i64,u32,bool
0,1517085,1659304800025,"""clicks""",0,276,true
0,1563459,1659304904511,"""clicks""",1,276,true
0,1309446,1659367439426,"""clicks""",2,276,true
0,16246,1659367719997,"""clicks""",3,276,true
0,1781822,1659367871344,"""clicks""",4,276,true
…,…,…,…,…,…,…
0,1639229,1661377507795,"""clicks""",188,276,true
0,1624436,1661377547329,"""clicks""",189,276,true
0,738987,1661377571633,"""clicks""",190,276,true


Hidden:


session,aid,ts,type,event_pos,session_len,is_observed
i64,i64,i64,str,i64,u32,bool
0,190818,1661547489220,"""clicks""",193,276,false
0,1157411,1661547534713,"""clicks""",194,276,false
0,138431,1661547574171,"""clicks""",195,276,false
0,543308,1661547622300,"""clicks""",196,276,false
0,1760145,1661547680429,"""clicks""",197,276,false
…,…,…,…,…,…,…
0,843110,1661684298768,"""clicks""",271,276,false
0,938007,1661684355390,"""clicks""",272,276,false
0,1228848,1661684528943,"""clicks""",273,276,false


Ground truth:


session,click_target,cart_targets,order_targets
i64,i64,list[i64],list[i64]
0,190818,"[275288, 974651, … 315914]","[1199474, 543308]"


In [8]:
#Recall@20
def recall_at_k(predictions, targets, k=20):
    if targets is None or len(targets) == 0:
        return None

    preds = predictions[:k]

    hits = len(set(preds) & set(targets))

    return hits / len(set(targets))

In [9]:
print(
    recall_at_k(
        predictions=[10, 20, 30, 40],
        targets=[20, 40],
        k=20
    )
)

print(
    recall_at_k(
        predictions=[10, 30],
        targets=[20, 40],
        k=20
    )
)

1.0
0.0


In [10]:
print(
    recall_at_k(
        predictions=[10, 20],
        targets=[20, 40],
        k=20
    )
)

0.5


In [11]:
global_popularity = (
    observed
    .group_by(["type", "aid"])
    .agg(
        pl.len().alias("count")
    )
    .sort(
        ["type", "count"],
        descending=[False, True]
    )
)

In [13]:
top_clicks = (
    global_popularity
    .filter(pl.col("type") == "clicks")
    .head(20)["aid"]
    .to_list()
)

top_carts = (
    global_popularity
    .filter(pl.col("type") == "carts")
    .head(20)["aid"]
    .to_list()
)

top_orders = (
    global_popularity
    .filter(pl.col("type") == "orders")
    .head(20)["aid"]
    .to_list()
)

print("Top clicks:", top_clicks)
print("Top carts:", top_carts)
print("Top orders:", top_orders)


Top clicks: [29735, 832192, 108125, 80222, 1733943, 1083665, 554660, 1743151, 1460571, 166037, 1498443, 247240, 1603001, 476629, 610733, 332654, 803928, 184976, 1680387, 1645990]
Top carts: [29735, 80222, 1672890, 1733943, 832192, 166037, 1498443, 1022566, 152547, 1083665, 485256, 1629608, 1743151, 544144, 351335, 244808, 554660, 332654, 673407, 923948]
Top orders: [80222, 351335, 1022566, 1733943, 166037, 1629608, 1083665, 332654, 29735, 923948, 832192, 326904, 1825743, 508883, 247240, 842805, 480578, 544144, 673407, 125278]


In [14]:
#check with the dumbest model
click_recalls = []

for row in ground_truth.iter_rows(named=True):
    target = row["click_target"]

    if target is None:
        continue

    score = recall_at_k(
        predictions=top_clicks,
        targets=[target],
        k=20
    )

    click_recalls.append(score)

global_click_recall = np.mean(click_recalls)

print("Global Popularity Click Recall@20:", global_click_recall)

Global Popularity Click Recall@20: 0.009910036547652516


In [15]:
cart_recalls = []

for row in ground_truth.iter_rows(named=True):
    targets = row["cart_targets"]

    if targets is None or len(targets) == 0:
        continue

    score = recall_at_k(
        predictions=top_carts,
        targets=targets,
        k=20
    )

    cart_recalls.append(score)

global_cart_recall = np.mean(cart_recalls)

print("Global Popularity Cart Recall@20:", global_cart_recall)

Global Popularity Cart Recall@20: 0.012679506657059894


In [16]:
order_recalls = []

for row in ground_truth.iter_rows(named=True):
    targets = row["order_targets"]

    if targets is None or len(targets) == 0:
        continue

    score = recall_at_k(
        predictions=top_orders,
        targets=targets,
        k=20
    )

    order_recalls.append(score)

global_order_recall = np.mean(order_recalls)

print("Global Popularity Order Recall@20:", global_order_recall)


Global Popularity Order Recall@20: 0.013696703084723918


In [17]:
#weighted score
global_weighted_score = (
    0.10 * global_click_recall
    + 0.30 * global_cart_recall
    + 0.60 * global_order_recall
)

print("Weighted OTTO Score:", global_weighted_score)

Weighted OTTO Score: 0.01301287750271757


In [18]:
results = []

results.append({
    "model": "global_popularity",
    "click_recall@20": global_click_recall,
    "cart_recall@20": global_cart_recall,
    "order_recall@20": global_order_recall,
    "weighted_score": global_weighted_score,
})

results_df = pl.DataFrame(results)

results_df

model,click_recall@20,cart_recall@20,order_recall@20,weighted_score
str,f64,f64,f64,f64
"""global_popularity""",0.00991,0.01268,0.013697,0.013013


In [19]:
recent_predictions = (
    observed
    .sort(["session", "ts"], descending=[False, True])
    .group_by("session", maintain_order=True)
    .agg(
        pl.col("aid")
        .unique(maintain_order=True)
        .head(20)
        .alias("recent_items")
    )
)

recent_predictions.head()

session,recent_items
i64,list[i64]
0,"[102416, 1436439, … 1813509]"
1,"[50049, 711125, … 1492293]"
2,"[78519, 1577398, … 763743]"
3,"[812246, 332817, … 254870]"
4,"[917213, 1554752, … 613619]"


In [20]:
recent_eval = (
    ground_truth
    .join(
        recent_predictions,
        on="session",
        how="inner"
    )
)

recent_eval.head()

session,click_target,cart_targets,order_targets,recent_items
i64,i64,list[i64],list[i64],list[i64]
0,190818,"[275288, 974651, … 315914]","[1199474, 543308]","[102416, 1436439, … 1813509]"
1,105393,[105393],[],"[50049, 711125, … 1492293]"
2,1605583,[],[],"[78519, 1577398, … 763743]"
3,822461,"[812246, 1638009, … 925352]","[1018433, 54857]","[812246, 332817, … 254870]"
4,758750,"[917213, 758750]",[],"[917213, 1554752, … 613619]"


In [21]:
click_scores = []
cart_scores = []
order_scores = []

for row in recent_eval.iter_rows(named=True):

    preds = row["recent_items"]

    click_target = row["click_target"]
    if click_target is not None:
        click_scores.append(
            recall_at_k(
                preds,
                [click_target],
                k=20
            )
        )

    cart_targets = row["cart_targets"]
    if cart_targets is not None and len(cart_targets) > 0:
        cart_scores.append(
            recall_at_k(
                preds,
                cart_targets,
                k=20
            )
        )

    order_targets = row["order_targets"]
    if order_targets is not None and len(order_targets) > 0:
        order_scores.append(
            recall_at_k(
                preds,
                order_targets,
                k=20
            )
        )

recency_click_recall = np.mean(click_scores)
recency_cart_recall = np.mean(cart_scores)
recency_order_recall = np.mean(order_scores)

recency_weighted_score = (
    0.10 * recency_click_recall
    + 0.30 * recency_cart_recall
    + 0.60 * recency_order_recall
)

print("Click Recall@20:", recency_click_recall)
print("Cart Recall@20:", recency_cart_recall)
print("Order Recall@20:", recency_order_recall)
print("Weighted score:", recency_weighted_score)

Click Recall@20: 0.34051568336077753
Cart Recall@20: 0.2769220863629735
Order Recall@20: 0.45453391141144406
Weighted score: 0.3898485410918362


In [22]:
results.append({
    "model": "session_recency",
    "click_recall@20": recency_click_recall,
    "cart_recall@20": recency_cart_recall,
    "order_recall@20": recency_order_recall,
    "weighted_score": recency_weighted_score,
})

results_df = pl.DataFrame(results)

results_df

model,click_recall@20,cart_recall@20,order_recall@20,weighted_score
str,f64,f64,f64,f64
"""global_popularity""",0.00991,0.01268,0.013697,0.013013
"""session_recency""",0.340516,0.276922,0.454534,0.389849


In [23]:
def otto_recall_at_20(
    eval_df,
    prediction_col,
    target_col,
    scalar_target=False
):
    total_hits = 0
    total_targets = 0

    for row in eval_df.iter_rows(named=True):
        predictions = row[prediction_col]

        if predictions is None:
            continue

        predictions = list(dict.fromkeys(predictions[:20]))

        if scalar_target:
            target = row[target_col]

            if target is None:
                continue

            targets = [target]

        else:
            targets = row[target_col]

            if targets is None or len(targets) == 0:
                continue

            targets = list(set(targets))

        total_hits += len(set(predictions) & set(targets))
        total_targets += min(20, len(targets))

    return total_hits / total_targets

In [24]:
recency_click_recall = otto_recall_at_20(
    recent_eval,
    prediction_col="recent_items",
    target_col="click_target",
    scalar_target=True
)

recency_cart_recall = otto_recall_at_20(
    recent_eval,
    prediction_col="recent_items",
    target_col="cart_targets"
)

recency_order_recall = otto_recall_at_20(
    recent_eval,
    prediction_col="recent_items",
    target_col="order_targets"
)

recency_weighted_score = (
    0.10 * recency_click_recall
    + 0.30 * recency_cart_recall
    + 0.60 * recency_order_recall
)

print("Click Recall@20:", recency_click_recall)
print("Cart Recall@20:", recency_cart_recall)
print("Order Recall@20:", recency_order_recall)
print("Weighted:", recency_weighted_score)

Click Recall@20: 0.34051568336077753
Cart Recall@20: 0.20124540349176578
Order Recall@20: 0.41869315796837836
Weighted: 0.3456410841646345


In [25]:
global_eval = ground_truth.with_columns(
    pl.lit(top_clicks).alias("click_predictions"),
    pl.lit(top_carts).alias("cart_predictions"),
    pl.lit(top_orders).alias("order_predictions"),
)

In [26]:
global_click_recall = otto_recall_at_20(
    global_eval,
    "click_predictions",
    "click_target",
    scalar_target=True
)

global_cart_recall = otto_recall_at_20(
    global_eval,
    "cart_predictions",
    "cart_targets"
)

global_order_recall = otto_recall_at_20(
    global_eval,
    "order_predictions",
    "order_targets"
)

global_weighted_score = (
    0.10 * global_click_recall
    + 0.30 * global_cart_recall
    + 0.60 * global_order_recall
)

print(global_click_recall)
print(global_cart_recall)
print(global_order_recall)
print(global_weighted_score)

0.009910036547652516
0.009019165727170236
0.00934657898418917
0.009304700763429825


In [32]:
dev = pl.read_parquet("../data/processed/train_dev.parquet")
dev_lengths = (
    dev.group_by("session")
    .agg(pl.len().alias("n_events"))
)

dev_lengths.select("n_events").describe()

statistic,n_events
str,f64
"""count""",100000.0
"""null_count""",0.0
"""mean""",17.00815
"""std""",34.162754
"""min""",2.0
"""25%""",3.0
"""50%""",6.0
"""75%""",15.0
"""max""",494.0


In [33]:
df = pl.read_parquet("../data/processed/train_dev.parquet")